## What is a Scikit learn Pipeline?
    Pipeline chain together multiple so that  the output of each step is used as the input to the next step.

    Pipelines makes it easy to use the same preprocessing to train and test .

REFERENCE VIDEO :- http://youtube.com/watch?v=xOccYkgRV4Q

In [110]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [111]:
df  = pd.read_csv('Titanic-Dataset.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [112]:
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin','Pclass'])

df.sample(5)

,Survived,Sex,Age,SibSp,Parch,Fare,Embarked
610,0,female,39.0,1,5,31.2750,S
110,0,male,47.0,0,0,52.0000,S
21,1,male,34.0,0,0,13.0000,S
300,1,female,NaN,0,0,7.7500,Q
310,1,female,24.0,0,0,83.1583,C


In [113]:
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=["Survived"]),df['Survived'],test_size = 0.2,random_state=42)

In [114]:
df.Age.isnull().sum()


np.int64(177)

In [115]:
df.Embarked.isnull().sum()


np.int64(2)

In [116]:
trf1 = ColumnTransformer(transformers=[
    ('age_Imputer',SimpleImputer(),[2]),
    ('embarked_Imputer',SimpleImputer(strategy='most_frequent'),[5])
], remainder= 'passthrough'
                         )

In [117]:
df.Sex.isnull().sum()

np.int64(0)

In [118]:
df.Embarked.unique()

array(['S', 'C', 'Q', nan], dtype=object)

In [119]:
df.sample(5)

,Survived,Sex,Age,SibSp,Parch,Fare,Embarked
477,0,male,29.0,1,0,7.0458,S
392,0,male,28.0,2,0,7.9250,S
85,1,female,33.0,3,0,15.8500,S
527,0,male,NaN,0,0,221.7792,S
502,0,female,NaN,0,0,7.6292,Q


In [ ]:
# still it is acategorical value so we need to encode it using one hot encoding as there is no order 

trf2 = ColumnTransformer(transformers=[
    ('sex_embarked_One-hot_encoder',OneHotEncoder(sparse_output=False),[1,5])
])

In [121]:
from sklearn.preprocessing import MinMaxScaler

trf3 = ColumnTransformer(transformers=[
    ('scaler',MinMaxScaler(),slice(0,8))  # we have 8 columns after one hot encoding so we need to scale all the columns slice func will do that for all columns ie applying the min max sccaler 
])

In [122]:
# you could complete the  pipeline here or you could add a classifier or regressor to the pipeline and then complete it and you can also add a feature selection step in this pipeline 

# for this i will be adding the classifier to the pipeline and then complete it 

trf4 = DecisionTreeClassifier()

In [123]:
pipeline = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
])

# Pipeline Vs make_pipeline

### the difference is the syntax length the make_pipeline is less than the Pipeline as Pipeline calls the Pipeline class and the make_pipeline uses the method

## When to use fit_transform() and when to use fit() in pipeline creation 

### the fit_transform() is used when the pipeline only consists of the preprocessing steps like in our current example but if we remove the classifier 


### the fit() is used when the pipeline consists of the preprocessing and also the model ie classifier or regressor 

In [124]:
pipeline.fit(x_train,y_train)

ValueError: all features must be in [0, 5] or [-6, 0]

### Explore the Pipeline


In [ ]:
pipeline.named_steps # The dict below will help to understand the pipeline the key are the name that we have given when creating the Pipline using the class name

# pipeline = Pipeline([
#     ('trf1',trf1),
#     ('trf2',trf2),
#     ('trf3',trf3),
#     ('trf4',trf4),
# ])   the 'trf1'  is used to understand the pipeline properly 


{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('age_Imputer', SimpleImputer(), [2]),
                                 ('embarked_Imputer',
                                  SimpleImputer(strategy='most_frequent'),
                                  [5])]),
 'trf2': ColumnTransformer(transformers=[('sex_embarked_One-hot_encoder',
                                  OneHotEncoder(sparse_output=False), [1, 5])]),
 'trf3': ColumnTransformer(transformers=[('scaler', MinMaxScaler(), slice(0, 8, None))]),
 'trf4': DecisionTreeClassifier()}

In [ ]:
## you can explore the pipeline as follows suppose you want to see the the mean value of the simple imputer then folow the steps i will show one by one so that you knwo how to approach any pipeline to get the value you want 

In [ ]:
pipeline.named_steps['trf1']

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('age_Imputer', ...), ('embarked_Imputer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` 

In [ ]:
pipeline.named_steps['trf1'].transformers_ # this will give you the list of transformers that we have in the trf1

[('age_Imputer', SimpleImputer(), [2]),
 ('embarked_Imputer', SimpleImputer(strategy='most_frequent'), [5]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 3, 4])]

In [ ]:
pipeline.named_steps['trf1'].transformers_[0]

('age_Imputer', SimpleImputer(), [2])

In [ ]:
pipeline.named_steps['trf1'].transformers_[0][1]
# this will give you the mean value of the age column that we have in the trf1 transformer 0 as we have two transformers in the trf1 transformer and the first one is for age column and the second one is for embarked column so we are accessing the first transformer using the index 0 and then we are accessing the stats_ attribute to get the mean value of the age column that we have in the trf1 transformer 0

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [ ]:
pipeline.named_steps['trf1'].transformers_[0][1].statistics_

array([0.55337079])

In [ ]:
# same will be done for the embarked in one step 

pipeline.named_steps['trf1'].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [ ]:
x_train.sample(5)

,Sex,Age,SibSp,Parch,Fare,Embarked
272,female,41.0,0,1,19.500,S
13,male,39.0,1,5,31.275,S
617,female,26.0,1,0,16.100,S
825,male,NaN,0,0,6.950,Q
21,male,34.0,0,0,13.000,S


In [ ]:
x_test.sample(5)

,Sex,Age,SibSp,Parch,Fare,Embarked
584,male,NaN,0,0,8.7125,C
33,male,66.0,0,0,10.5000,S
65,male,NaN,1,1,15.2458,C
280,male,65.0,0,0,7.7500,Q
830,female,15.0,1,0,14.4542,C


In [ ]:
# now use the pipeline to predict 

y_pred = pipeline.predict(x_test)

ValueError: Found unknown categories [7.0458, 7.875, 9.225, 7.7292, 8.7125, 12.875, 8.1583, 8.4333, 15.55, 9.8458, 8.4583, 25.925, 26.2833, 26.3875, 29.0, 30.6958, 32.3208, 32.5, 34.6542, 7.7875, 38.5, 39.4, 49.5, 56.9292, 61.9792, 63.3583, 76.2917, 221.7792] in column 1 during transform